# 16종 동화 캐릭터 SDXL LoRA 학습 및 검증

이 노트북은 16종 캐릭터(`female_01~08`, `male_01~08`) 144장의 정제된 데이터셋(`character_lora_dataset.zip`)을 활용하여 SDXL LoRA를 학습하고, 캐릭터 일관성 및 소품 제어(망원경 1개 등)를 검증합니다.

- Colab 메뉴에서 **런타임 > 런타임 유형 변경 > T4 GPU** (또는 A100/L4)를 선택하세요.
- Hugging Face 토큰은 Colab **보안 비밀(Secrets)**에 `HF_TOKEN`으로 등록하면 편리합니다.
- 결과 가중치는 Google Drive의 `MyDrive/soft_capston_training/`에 자동 보관됩니다.
- **데이터셋 연결**: `character_lora_dataset.zip`(24.76MB)을 Google Drive의 `soft_capston_training/`에 넣거나, Colab 파일 탭에 직접 업로드하면 자동 인식됩니다.

> **모델 호환성 안내**: 이 LoRA는 `stabilityai/stable-diffusion-xl-base-1.0` 전용 가중치입니다. 기존 FLUX.2-klein 시험용 Space에는 로드할 수 없으며, 서빙 시 SDXL Diffusers 파이프라인이나 SDXL Space를 사용합니다.

In [ ]:
# 1. 학습 설정
REPO_URL = "https://github.com/Leejinhoe/soft_capston.git"
REPO_BRANCH = "main"
DIFFUSERS_TAG = "v0.38.0"

BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
DRIVE_FOLDER = "soft_capston_training"
USE_GOOGLE_DRIVE = True           # 실패 시 /content로 자동 전환
ALLOW_LOCAL_STORAGE_FALLBACK = True

TRAIN_STEPS = 800                 # 빠른 검증: 600~800 steps (T4 기준 약 25~35분)
RESOLUTION = 512                  # T4 안전 모드: 512 (A100/L4 또는 고사양 시 768/1024 가능)
LORA_RANK = 16
LEARNING_RATE = 1e-4
RESUME_FROM_LATEST = True
REBUILD_DATASET = True
SEED = 42

# 특정 캐릭터만 선택 학습하려면 키를 지정하세요 (예: ["male_01", "female_01"]).
# 빈 리스트([])이면 16종 캐릭터 전체를 학습합니다.
TRAIN_ONLY = []


In [ ]:
# 2. GPU 확인 및 공식 Diffusers 학습 환경 설치
import os, shutil, subprocess, sys
from pathlib import Path

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print("GPU:", gpu_name)
if "T4" not in gpu_name and "L4" not in gpu_name and "A100" not in gpu_name:
    print("주의: GPU 종류가 예상과 다릅니다. VRAM이 16GB 미만이면 OOM이 발생할 수 있습니다.")

# 저장소 폴더를 diffusers로 만들면 /content가 Python 경로에 있어 패키지 이름과 충돌합니다.
legacy_diffusers_dir = Path("/content/diffusers")
if (legacy_diffusers_dir / ".git").exists():
    shutil.rmtree(legacy_diffusers_dir)
DIFFUSERS_DIR = Path("/content/hf_diffusers_repo")
if not DIFFUSERS_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", DIFFUSERS_TAG,
        "https://github.com/huggingface/diffusers.git", str(DIFFUSERS_DIR)
    ], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.38.0", "accelerate>=1.2.0",
    "transformers>=4.46.0,<5", "datasets>=3.1.0",
    "peft>=0.14.0", "bitsandbytes>=0.45.0", "xformers", "safetensors",
    "huggingface_hub>=0.27.0", "Pillow>=10.0.0"
], check=True)

# 이전 실패 셀에서 만들어진 잘못된 모듈 캐시를 제거합니다.
for module_name in list(sys.modules):
    if module_name == "diffusers" or module_name.startswith("diffusers."):
        del sys.modules[module_name]
import diffusers
from diffusers import AutoPipelineForText2Image
if diffusers.__version__ != "0.38.0":
    raise RuntimeError(f"Diffusers 버전이 잘못되었습니다: {diffusers.__version__}")
print("Diffusers:", diffusers.__version__, diffusers.__file__)

from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")
print("환경 설치 완료")

In [ ]:
# 3. Google Drive, Hugging Face, 프로젝트 연결
import getpass, subprocess
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import login

storage_base = Path("/content")
drive_connected = False
if USE_GOOGLE_DRIVE:
    try:
        drive.mount("/content/drive", force_remount=False, timeout_ms=120000)
        storage_base = Path("/content/drive/MyDrive")
        drive_connected = True
        print("Google Drive 마운트 성공:", storage_base)
    except Exception as exc:
        print("Google Drive 연결 실패 (임시 저장소 사용):", exc)
        if not ALLOW_LOCAL_STORAGE_FALLBACK:
            raise

DRIVE_ROOT = storage_base / DRIVE_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if not hf_token:
    hf_token = getpass.getpass("Hugging Face Token 입력: ").strip()
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face 인증 완료")
else:
    print("주의: HF_TOKEN이 입력되지 않았습니다. 공개 모델만 다운로드 가능합니다.")

PROJECT_DIR = Path("/content/soft_capston")
if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
        REPO_URL, str(PROJECT_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

print("프로젝트 디렉토리:", PROJECT_DIR)
print("결과/데이터 폴더:", DRIVE_ROOT)


In [ ]:
# 4. 16종 동화 캐릭터 + 클래식 5대 주인공 트리거 토큰 및 메타데이터 정의
CHARACTERS = {
    # 16종 기본 캐릭터 (male_01~08, female_01~08)
    "male_01": {"token": "sks_male_01", "name": "민호", "description": "Korean boy around age 8 with a round face, short black hair, bright brown eyes, a cobalt tunic, red scarf, leather belt, and small silver sword."},
    "male_02": {"token": "sks_male_02", "name": "준", "description": "Korean teenage boy with tousled dark hair, amber eyes, an ochre field jacket, navy trousers, canvas satchel, brass compass, and sturdy brown boots."},
    "male_03": {"token": "sks_male_03", "name": "태산", "description": "Korean adult man with a broad kind face, tied-back black hair, a moss-green cloak, bronze shoulder guard, dark tunic, and carved wooden staff."},
    "male_04": {"token": "sks_male_04", "name": "도윤", "description": "Elderly Korean man with silver hair, short white beard, gentle wrinkles, a brown wool cap, forest-green vest, patched trousers, and an old rolled map."},
    "male_05": {"token": "sks_male_05", "name": "보리", "description": "Korean boy around age 7 with fluffy dark-brown hair, freckles, a leaf-green capelet, cream overalls, orange boots, and an acorn-shaped pouch."},
    "male_06": {"token": "sks_male_06", "name": "레이븐", "description": "Young Korean man with a sharp pale face, long ink-black hair, gray eyes, a black-violet royal coat, silver crown, high boots, and a dark crystal ring."},
    "male_07": {"token": "sks_male_07", "name": "이안", "description": "Korean boy around age 9 with neatly parted chestnut hair, warm brown eyes, an ivory prince jacket, sky-blue sash, gold trim, and white ankle boots."},
    "male_08": {"token": "sks_male_08", "name": "하늘", "description": "Korean adult man with a slender face, wavy midnight hair, blue-gray eyes, a teal star-patterned robe, moon brooch, leather spellbook, and crystal wand."},
    "female_01": {"token": "sks_female_01", "name": "미나", "description": "Korean girl around age 8 with a round friendly face, short dark-brown bob, midnight-blue star cape, lavender tunic, brown boots, and glowing star wand."},
    "female_02": {"token": "sks_female_02", "name": "하나", "description": "Korean girl around age 8 with two low braids, bright brown eyes, a sunflower-yellow cardigan, denim-blue overalls, coral shoes, and friendship bracelet."},
    "female_03": {"token": "sks_female_03", "name": "미란", "description": "Korean teenage girl with a soft oval face, long chestnut hair, hazel eyes, a rose-pink royal dress, pearl circlet, cream cape, and golden key necklace."},
    "female_04": {"token": "sks_female_04", "name": "루나", "description": "Young Korean woman with a heart-shaped face, wavy silver-brown hair, green eyes, a leaf-layered emerald dress, translucent wings, vine belt, and glowing lantern."},
    "female_05": {"token": "sks_female_05", "name": "서연", "description": "Korean adult woman with a confident oval face, black hair in a low ponytail, an amber scarf, navy expedition coat, charcoal trousers, notebook, and magnifying glass."},
    "female_06": {"token": "sks_female_06", "name": "아린", "description": "Elderly Korean woman with silver hair in a low bun, warm smile lines, a plum shawl, cream hanbok-inspired dress, herb pouch, and carved willow cane."},
    "female_07": {"token": "sks_female_07", "name": "나라", "description": "Young Korean woman with an angular face, long raven hair, violet eyes, a burgundy-black royal gown, obsidian crown, silver shoulder cape, and dark mirror."},
    "female_08": {"token": "sks_female_08", "name": "솔", "description": "Korean teenage girl with a sun-kissed round face, wavy dark-brown hair, a moss cape, leaf-pattern cream tunic, forest boots, acorn pendant, and short bow."},
    # 클래식 5대 장르 주인공
    "fantasy_mina": {"token": "sks_fantasy_mina", "name": "클래식 미나", "description": "Korean girl around age 8 with a short dark-brown bob, midnight-blue star cape, lavender tunic, brown boots, glowing star wand."},
    "adventure_jun": {"token": "sks_adventure_jun", "name": "클래식 준", "description": "Korean teenage boy with tousled black hair, ochre field jacket, deep-red scarf, navy shorts, brown boots, brass compass."},
    "nature_sol": {"token": "sks_nature_sol", "name": "클래식 솔", "description": "Korean teenage girl with wavy dark-brown hair, moss-green capelet, leaf-pattern cream tunic, forest-brown boots, acorn pendant, wooden lantern."},
    "friendship_hana": {"token": "sks_friendship_hana", "name": "클래식 하나", "description": "Korean girl around age 8 with two low braids, sunflower-yellow cardigan, white blouse, denim-blue overalls, coral shoes."},
    "mystery_on": {"token": "sks_mystery_on", "name": "클래식 온", "description": "Korean boy around age 9 with dark hair, plum trench coat, charcoal trousers, brass pocket watch, magnifying glass."},
}
print(f"등록된 캐릭터: {len(CHARACTERS)}종 (기본 16종 + 클래식 5인방)")
for key, info in list(CHARACTERS.items())[:5]:
    print(f"  - [{info['token']}] {key} ({info['name']})")
print("  ...")


In [ ]:
# 5. Prepare shared imports. The ZIP already contains training images.
import gc, json, math, random, shutil, tempfile, zipfile
import torch
from pathlib import Path
from PIL import Image
from IPython.display import display
print('Background generation is not needed for this dataset.')


In [ ]:
# 6. character_lora_dataset.zip 자동 감지, 압축 해제 및 검증
import json, math, shutil, zipfile
from pathlib import Path
from PIL import Image
from IPython.display import display

DATASET_DIR = Path(tempfile.mkdtemp(prefix="character_lora_", dir="/content"))

# 1. 다중 경로 순차 검색
candidate_paths = [
    DRIVE_ROOT / "character_lora_dataset.zip",
    Path("/content/drive/MyDrive/soft_capston_training/character_lora_dataset.zip"),
    Path("/content/drive/MyDrive/character_lora_dataset.zip"),
    Path("/content/character_lora_dataset.zip"),
    PROJECT_DIR / "artifacts" / "character_lora_dataset.zip",
]

found_zip = None
for cand in candidate_paths:
    if cand.is_file():
        found_zip = cand
        print(f"[발견] LoRA 데이터셋 ZIP: {found_zip} ({found_zip.stat().st_size / 1024 / 1024:.2f} MB)")
        break

# 2. 파일이 없으면 Colab 업로드 다이얼로그 표시
if not found_zip:
    from google.colab import files
    print("\n[안내] Google Drive 또는 Colab에 character_lora_dataset.zip이 없습니다.")
    print("아래 '파일 선택' 버튼을 눌러 artifacts/character_lora_dataset.zip (24.76 MB)을 업로드하세요.")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        save_target = Path("/content") / Path(fname).name
        save_target.write_bytes(content)
        if save_target.suffix.lower() == ".zip":
            found_zip = save_target
            break

if not found_zip or not found_zip.is_file():
    raise FileNotFoundError("character_lora_dataset.zip을 찾을 수 없습니다. 파일을 업로드해주세요.")

# 3. 압축 해제
with zipfile.ZipFile(found_zip, "r") as archive:
    root = DATASET_DIR.resolve()
    for member in archive.infolist():
        if not (root / member.filename).resolve().is_relative_to(root):
            raise ValueError("Unsafe dataset ZIP path")
    archive.extractall(DATASET_DIR)

# 4. 메타데이터 및 이미지 확인
metadata_path = DATASET_DIR / "metadata.jsonl"
if not metadata_path.exists():
    raise FileNotFoundError(f"데이터셋 내 metadata.jsonl이 없습니다: {DATASET_DIR}")

records = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# TRAIN_ONLY 필터링 지원
if TRAIN_ONLY:
    records = [r for r in records if r.get("character_key") in TRAIN_ONLY]
    print(f"선택된 캐릭터 필터 적용: {TRAIN_ONLY} -> {len(records)}장")
    if not records:
        raise ValueError("TRAIN_ONLY selected no dataset records")
    selected = {r["file_name"] for r in records}
    for image_path in (DATASET_DIR / "images").glob("*.png"):
        if image_path.relative_to(DATASET_DIR).as_posix() not in selected:
            image_path.unlink()
            image_path.with_suffix(".txt").unlink(missing_ok=True)
    # metadata.jsonl 다시 쓰기
    with open(metadata_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"[데이터셋 준비 완료] 총 {len(records)}장 학습 데이터 확보")

# 5. 프리뷰 썸네일 표시
preview_images = []
for r in records[:12]:
    img_path = DATASET_DIR / r["file_name"]
    if img_path.exists():
        preview_images.append(Image.open(img_path).convert("RGB").resize((160, 160)))

if preview_images:
    cols = 4
    rows = math.ceil(len(preview_images) / cols)
    grid = Image.new("RGB", (cols * 160, rows * 160), "white")
    for idx, thumb in enumerate(preview_images):
        grid.paste(thumb, ((idx % cols) * 160, (idx // cols) * 160))
    display(grid)


In [ ]:
# 7. SDXL LoRA 학습 실행
import gc, subprocess, torch

OUTPUT_DIR = DRIVE_ROOT / "sdxl_fairytale_lora_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_script = DIFFUSERS_DIR / "examples" / "text_to_image" / "train_text_to_image_lora_sdxl.py"
if not train_script.exists():
    raise FileNotFoundError(f"Diffusers 학습 스크립트 누락: {train_script}")

command = [
    "accelerate", "launch",
    "--mixed_precision=fp16",
    str(train_script),
    f"--pretrained_model_name_or_path={BASE_MODEL}",
    f"--train_data_dir={DATASET_DIR}",
    "--caption_column=text",
    f"--resolution={RESOLUTION}",
    "--train_batch_size=1",
    "--gradient_accumulation_steps=4",
    "--gradient_checkpointing",
    "--enable_xformers_memory_efficient_attention",
    f"--learning_rate={LEARNING_RATE}",
    "--lr_scheduler=cosine",
    "--lr_warmup_steps=50",
    f"--max_train_steps={TRAIN_STEPS}",
    "--checkpointing_steps=200",
    "--checkpoints_total_limit=2",
    f"--rank={LORA_RANK}",
    "--snr_gamma=5.0",
    "--dataloader_num_workers=2",
    "--report_to=none",
    f"--seed={SEED}",
    "--mixed_precision=fp16",
    f"--output_dir={OUTPUT_DIR}",
]
if RESUME_FROM_LATEST and any(OUTPUT_DIR.glob("checkpoint-*")):
    command.append("--resume_from_checkpoint=latest")

print(f"학습 시작 (총 {TRAIN_STEPS} Steps). T4 기준 약 25~35분 소요 예상됩니다.")
print("실행 명령:", " ".join(map(str, command)))
subprocess.run(command, check=True)

weight_path = OUTPUT_DIR / "pytorch_lora_weights.safetensors"
if not weight_path.is_file():
    raise RuntimeError("LoRA 가중치가 생성되지 않았습니다. 학습 로그를 확인하세요.")
print(f"[학습 완료] 가중치 생성: {weight_path} ({weight_path.stat().st_size / 1024 / 1024:.1f} MB)")


In [ ]:
# 8. 학습 결과 정밀 추론 및 3대 핵심 액션 검증
import gc, torch
from PIL import Image
from IPython.display import display
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

gc.collect()
torch.cuda.empty_cache()

sample_pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
sample_pipe.scheduler = DPMSolverMultistepScheduler.from_config(sample_pipe.scheduler.config)
sample_pipe.load_lora_weights(str(OUTPUT_DIR), weight_name="pytorch_lora_weights.safetensors")
sample_pipe.enable_model_cpu_offload()
try:
    sample_pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

SAMPLE_DIR = OUTPUT_DIR / "samples"
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# 3대 핵심 액션 테스트 프롬프트 정의 (소품 환각 억제 및 동작 적응성 검증)
test_cases = [
    {
        "name": "male_01_single_telescope",
        "prompt": "sks_male_01 boy, looking through exactly one handheld brass telescope, one free hand, solid white background, polished watercolor fairytale storybook illustration style, rounded clean lines, soft cinematic lighting",
        "focus": "망원경이 1개만 나오는지, 얼굴/의상(파란 튜닉, 빨간 스카프) 유지 여부"
    },
    {
        "name": "female_01_reading_book",
        "prompt": "sks_female_01 girl, reading an open storybook quietly, sitting with gentle smile, solid white background, polished watercolor fairytale storybook illustration style, rounded clean lines, soft cinematic lighting",
        "focus": "책 읽기 포즈 자연스러움, 단발머리/별 망토 유지 여부"
    },
    {
        "name": "male_01_waving_hand",
        "prompt": "sks_male_01 boy, waving hand warmly with a bright cheerful smile, solid white background, polished watercolor fairytale storybook illustration style, rounded clean lines, soft cinematic lighting",
        "focus": "손 흔들기 동작 및 표정 변화"
    },
]

sample_images = []
for idx, tc in enumerate(test_cases):
    print(f"[추론 {idx+1}/3] {tc['name']} - 검증 포인트: {tc['focus']}")
    img = sample_pipe(
        prompt=tc["prompt"],
        negative_prompt="text, watermark, blurry, duplicate props, two telescopes, two heads, extra limbs, ugly, solid dark background",
        num_inference_steps=28,
        guidance_scale=6.5,
        width=RESOLUTION,
        height=RESOLUTION,
        generator=torch.Generator(device="cpu").manual_seed(SEED + idx),
    ).images[0]
    out_path = SAMPLE_DIR / f"{tc['name']}.png"
    img.save(out_path)
    print(f"  -> 저장: {out_path}")
    sample_images.append(img.resize((320, 320)))

# 그리드 출력
grid = Image.new("RGB", (len(sample_images) * 320, 320), "white")
for idx, img in enumerate(sample_images):
    grid.paste(img, (idx * 320, 0))
display(grid)

del sample_pipe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 9. 결과 메타데이터 저장 및 배포용 ZIP 패키징
import json, zipfile

config_path = OUTPUT_DIR / "fairytale_training_config.json"
config_path.write_text(json.dumps({
    "base_model": BASE_MODEL,
    "diffusers_tag": DIFFUSERS_TAG,
    "resolution": RESOLUTION,
    "train_steps": TRAIN_STEPS,
    "lora_rank": LORA_RANK,
    "learning_rate": LEARNING_RATE,
    "trigger_tokens": {key: val["token"] for key, val in CHARACTERS.items()},
}, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = DRIVE_ROOT / "fairytale_character_sdxl_lora_result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and "checkpoint-" not in str(path):
            archive.write(path, path.relative_to(OUTPUT_DIR))

print("최종 결과 ZIP 아카이브:", zip_path)
print("가중치 파일: pytorch_lora_weights.safetensors")
if not drive_connected:
    from google.colab import files
    print("Drive가 연결되지 않아 로컬 다운로드를 시작합니다.")
    files.download(str(zip_path))


## 결과 평가 및 서빙 연동 안내

### 1. 객관적 검증 기준 (가설 검증)
1. **캐릭터 정체성 유지율**: 원본 일러스트 대비 얼굴 이목구비, 헤어스타일, 대표 의상 색상이 유지되는지 확인합니다.
2. **소품 환각 억제 (Fail-closed)**: `male_01` 망원경 생성 시 손에 들린 망원경이 정확히 1개인지, 추가 소품이 배경에 떠다니지 않는지 검사합니다.
3. **새로운 포즈 적응성**: 학습 세트에 없던 '망원경 보기'나 '책 읽기' 등의 동작이 부자연스럽지 않게 융합되는지 확인합니다.

### 2. 백엔드 서빙 경로 분리 안내
> **주의**: 본 가중치는 SDXL LoRA이므로 시험용 FLUX.2-klein Space(`dnossdf/fairytale-multiref-test`)에 직접 로드할 수 없습니다.
- **서빙 방안**: 생성된 `pytorch_lora_weights.safetensors`는 전용 SDXL Hugging Face Space 또는 Diffusers 파이프라인에서 로드하여 서비스합니다.
- **안전 폴백**: 추론 실패 또는 검증 기준 미달 시, 앱은 사전에 준비된 고품질 정적 캐릭터 에셋(`character_assets.py`)으로 무중단 폴백(Fail-closed)합니다.
